# per-rank-cuda-device — faded example 3: Only rank 0 is master

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `per-rank-cuda-device`. Running the beacon reports progress on the `Distributed: per-rank cuda device` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: per-rank cuda device` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`per-rank-cuda-device`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "per-rank-cuda-device"
DD_SUBTOPIC = "Distributed: per-rank cuda device"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In a distributed job, rank 0 is designated as the single 'master' process responsible for operations that must happen exactly once: saving checkpoints, logging metrics, initializing datasets. The `is_master` flag is simply `rank == 0`. All other ranks have `is_master = False` and skip those side effects.

## Faded exercise 3

### Exercise — Only rank 0 is master

Complete `make_rank_info(rank, world_size)`. Return a dict with `device`, `is_master`, and `device_str`. The `is_master` field must be `True` only when `rank == 0`.

Fill in the is_master value.

**Fill in:** Assign is_master as True only when this rank is 0, False for all other ranks.

In [ ]:
import torch as t

def make_rank_info(rank: int, world_size: int) -> dict:
    device = t.device(f'cuda:{rank}')
    is_master = None  # TODO: Assign is_master as True only when this rank is 0, False for all other ranks.
    return {
        'device': device,
        'is_master': is_master,
        'device_str': f'cuda:{rank}',
    }

for r in range(3):
    info = make_rank_info(r, 3)
    print(r, info['is_master'])


def _test():
    import torch as t
    for world_size in [2, 4, 8]:
        infos = [make_rank_info(r, world_size) for r in range(world_size)]
        masters = [i for i in infos if i['is_master']]
        assert len(masters) == 1, f'expected exactly 1 master, got {len(masters)}'
        assert infos[0]['is_master'] is True, 'rank 0 must be master'
        for i, info in enumerate(infos[1:], start=1):
            assert info['is_master'] is False, f'rank {i} must not be master'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def make_rank_info(rank: int, world_size: int) -> dict:
    device = t.device(f'cuda:{rank}')
    is_master = rank == 0
    return {
        'device': device,
        'is_master': is_master,
        'device_str': f'cuda:{rank}',
    }

for r in range(3):
    info = make_rank_info(r, 3)
    print(r, info['is_master'])
```
</details>